# Cibuco_Boriken — BirdCLEF+ 2026 Inference
**Team:** Cibuco_Boriken | **Backbone:** EfficientNet-B0 | **CFAR T=0.3**

In [ ]:
# ── Cell 1: Install dependencies ──
!pip install -q librosa audioread
print('Dependencies installed ✅')

In [ ]:
# ── Cell 2: Clone repo ──
!git clone https://github.com/drosadocastro-bit/cibuco-boriken
import sys
sys.path.insert(0, '/kaggle/working/cibuco-boriken')
print('Repo cloned ✅')

In [ ]:
# ── Cell 3: Imports ──
import os
import numpy as np
import pandas as pd
import torch
import librosa
from pathlib import Path
from tqdm import tqdm

# Paths
DATA_DIR    = Path('/kaggle/input/birdclef-2026')
MODEL_PATH  = Path('/kaggle/input/cibuco-boriken-model/efficientnet_b0_33k_best.pt')
OUTPUT_PATH = Path('/kaggle/working/submission.csv')

# Config
SAMPLE_RATE  = 32000
WINDOW_SEC   = 5.0
HOP_SEC      = 2.5       # 50% overlap
N_MELS       = 128
TEMPERATURE  = 0.3       # CFAR Goldilocks zone
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {DEVICE}')
print(f'Data:   {DATA_DIR}')
print(f'Model:  {MODEL_PATH}')
print('Imports complete ✅')

In [ ]:
# ── Cell 4: Load species list from train.csv ──
train_df = pd.read_csv(DATA_DIR / 'train.csv')
SPECIES = sorted(train_df['primary_label'].unique().tolist())
NUM_SPECIES = len(SPECIES)
print(f'Species: {NUM_SPECIES}')
print(f'First 5: {SPECIES[:5]}')

In [ ]:
# ── Cell 5: Load model ──
import torchvision.models as tv_models
import torch.nn as nn

class BirdClassifier(nn.Module):
    def __init__(self, num_classes, temperature=0.3):
        super().__init__()
        self.temperature = temperature
        backbone = tv_models.efficientnet_b0(weights=None)
        in_features = backbone.classifier[1].in_features
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.classifier = nn.Linear(in_features, num_classes)

    def forward(self, x):
        # Convert 1-ch mel to 3-ch
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)
        features = self.backbone(x)
        return self.classifier(features)

    def predict(self, x):
        with torch.no_grad():
            logits = self.forward(x)
            return torch.sigmoid(logits / self.temperature)

model = BirdClassifier(num_classes=NUM_SPECIES, temperature=TEMPERATURE)
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
# Handle various checkpoint formats
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
elif isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
    model.load_state_dict(checkpoint['state_dict'])
else:
    model.load_state_dict(checkpoint)

model = model.to(DEVICE)
model.eval()
print(f'Model loaded on {DEVICE} ✅')

In [ ]:
# ── Cell 6: Audio → mel-spectrogram ──
def audio_to_mel(audio, sr=SAMPLE_RATE, n_mels=N_MELS):
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=n_mels,
        n_fft=1024, hop_length=320, fmin=50, fmax=14000
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    # Normalize to [0, 1]
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
    return mel_db.astype(np.float32)

def load_audio_windows(filepath, window_sec=WINDOW_SEC, hop_sec=HOP_SEC, sr=SAMPLE_RATE):
    """Load audio and split into overlapping windows. Returns (windows, timestamps)."""
    try:
        audio, _ = librosa.load(filepath, sr=sr, mono=True)
    except Exception as e:
        print(f'Error loading {filepath}: {e}')
        return [], []

    window_samples = int(window_sec * sr)
    hop_samples    = int(hop_sec * sr)
    windows, timestamps = [], []

    start = 0
    while start + window_samples <= len(audio):
        chunk = audio[start:start + window_samples]
        windows.append(chunk)
        timestamps.append(start / sr)  # seconds
        start += hop_samples

    # Last partial window — pad if needed
    if start < len(audio):
        chunk = audio[start:]
        chunk = np.pad(chunk, (0, window_samples - len(chunk)))
        windows.append(chunk)
        timestamps.append(start / sr)

    return windows, timestamps

print('Audio utilities ready ✅')

In [ ]:
# ── Cell 7: CFAR adaptive thresholding ──
def cfar_threshold(all_probs, k=2.0, floor=0.05, ceiling=0.95):
    """
    Per-species CFAR thresholding.
    all_probs: (N_windows, N_species) numpy array
    Returns: thresholds (N_species,)
    """
    mu    = all_probs.mean(axis=0)   # per-species mean
    sigma = all_probs.std(axis=0)    # per-species std
    thresholds = mu + k * sigma
    thresholds = np.clip(thresholds, floor, ceiling)
    return thresholds

print('CFAR utilities ready ✅')

In [ ]:
# ── Cell 8: Run inference on all test soundscapes ──
soundscape_dir = DATA_DIR / 'test_soundscapes'
soundscape_files = sorted(soundscape_dir.glob('*.ogg'))
print(f'Test soundscapes: {len(soundscape_files)}')

BATCH_SIZE = 32
all_rows = []

for sf in tqdm(soundscape_files, desc='Inference'):
    file_id = sf.stem  # e.g. 'soundscape_12345'

    # Load windows
    windows, timestamps = load_audio_windows(str(sf))
    if not windows:
        continue

    # Mel features for all windows
    mels = np.stack([audio_to_mel(w) for w in windows])  # (N, n_mels, T)
    mels = mels[:, np.newaxis, :, :]                      # (N, 1, n_mels, T)
    mels_t = torch.from_numpy(mels).to(DEVICE)

    # Batch inference
    probs_list = []
    for i in range(0, len(mels_t), BATCH_SIZE):
        batch = mels_t[i:i+BATCH_SIZE]
        probs_list.append(model.predict(batch).cpu().numpy())
    all_probs = np.vstack(probs_list)  # (N_windows, N_species)

    # CFAR adaptive thresholds from this file's activation distribution
    thresholds = cfar_threshold(all_probs, k=2.0)

    # Build submission rows — one per 5-second window
    for idx, (probs, ts) in enumerate(zip(all_probs, timestamps)):
        # Row ID format: {file_id}_{end_time_seconds}
        end_time = int(ts + WINDOW_SEC)
        row_id = f'{file_id}_{end_time}'

        # Raw probabilities for ROC-AUC scoring (DO NOT threshold for submission)
        row = {'row_id': row_id}
        for species, prob in zip(SPECIES, probs):
            row[species] = float(prob)
        all_rows.append(row)

print(f'Total rows: {len(all_rows)} ✅')

In [ ]:
# ── Cell 9: Generate submission.csv ──
submission_df = pd.DataFrame(all_rows)

# Align columns to sample_submission.csv
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
expected_cols = sample_sub.columns.tolist()

# Add any missing species columns as 0
for col in expected_cols:
    if col not in submission_df.columns:
        submission_df[col] = 0.0

# Reorder to match sample submission
submission_df = submission_df[expected_cols]

# Fill any NaN
submission_df = submission_df.fillna(0.0)

submission_df.to_csv(OUTPUT_PATH, index=False)
print(f'Submission saved: {OUTPUT_PATH} ✅')
print(f'Shape: {submission_df.shape}')
print(submission_df.head(3))

In [ ]:
# ── Cell 10: Validate submission ──
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
our_sub    = pd.read_csv(OUTPUT_PATH)

print('=== Submission Validation ===')
print(f'Expected rows:  {len(sample_sub)}')
print(f'Our rows:       {len(our_sub)}')
print(f'Expected cols:  {len(sample_sub.columns)}')
print(f'Our cols:       {len(our_sub.columns)}')
print(f'Columns match:  {set(sample_sub.columns) == set(our_sub.columns)}')
print(f'Prob range:     [{our_sub[SPECIES[0]].min():.4f}, {our_sub[SPECIES[0]].max():.4f}]')
print()
if len(our_sub) == len(sample_sub):
    print('✅ SUBMISSION VALID — ready to submit!')
else:
    print('⚠️  Row count mismatch — check row_id format')